# Syllabus Structured Extractor Training (Colab)

For the **full** pipeline from `us_freshman_core_syllabi_index.csv` (CSV → URLs → ingest → process → build → train), use [`run_full_syllabus_pipeline.ipynb`](run_full_syllabus_pipeline.ipynb).

This notebook builds fine-tuning data from an **ingested** syllabus JSONL file (text already in the file), trains a Hugging Face model with LoRA, tracks the run with MLflow, and shows a sample inference result.

## 1. Runtime setup

Use a GPU runtime in Colab if one is available.

In [ ]:
!nvidia-smi || true
!pip -q install -U pip
!pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip -q install accelerate datasets mlflow peft transformers trl pymupdf requests duckduckgo-search pytest

## 2. Get the repository into Colab

Set `REPO_URL` if you want to clone from GitHub. If the notebook is already inside the repo, skip the clone cell.

In [ ]:
import os
from pathlib import Path

REPO_URL = os.environ.get("REPO_URL", "")
REPO_DIR = Path("/content/training_pipeline")

if REPO_URL and not REPO_DIR.exists():
    !git clone "$REPO_URL" "$REPO_DIR"

if not REPO_DIR.exists():
    raise FileNotFoundError("Set REPO_URL or place this notebook inside the training_pipeline repo in Colab.")

%cd /content/training_pipeline
!ls -la

## 3. Upload or point to your ingested JSONL

The input should contain one JSON object per line and include one text field such as `text`, `content`, or `body`.

In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
if not uploaded:
    raise ValueError("Upload your ingested JSONL file to continue.")

INPUT_JSONL = Path(next(iter(uploaded.keys()))).resolve()
print("Using input:", INPUT_JSONL)

## 4. Build fine-tuning JSONL files

In [ ]:
TRAIN_JSONL = "/content/training_pipeline/data/finetune/train.jsonl"
VALID_JSONL = "/content/training_pipeline/data/finetune/valid.jsonl"

!python build_finetune_dataset.py \
  --input-jsonl "$INPUT_JSONL" \
  --train-output "$TRAIN_JSONL" \
  --valid-output "$VALID_JSONL" \
  --validation-ratio 0.1

!python - <<'PY'
import json
from pathlib import Path

train_path = Path("/content/training_pipeline/data/finetune/train.jsonl")
sample = json.loads(train_path.read_text(encoding="utf-8").splitlines()[0])
print("Messages:", [m["role"] for m in sample["messages"]])
print(sample["messages"][2]["content"][:800])
PY

## 5. Train with Hugging Face + MLflow

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_DIR = "/content/training_pipeline/artifacts/hf_syllabus_extractor"
MLFLOW_TRACKING_URI = "file:///content/training_pipeline/mlruns"
MLFLOW_EXPERIMENT = "syllabus-structured-extraction"
MLFLOW_RUN_NAME = "colab-qwen25-structured-v1"
USE_BF16 = True
USE_FP16 = False

bf16_flag = "--bf16" if USE_BF16 else ""
fp16_flag = "--fp16" if USE_FP16 else ""

In [ ]:
!python train_hf_structured_extractor.py \
  --train-jsonl "$TRAIN_JSONL" \
  --valid-jsonl "$VALID_JSONL" \
  --model-name "$MODEL_NAME" \
  --output-dir "$OUTPUT_DIR" \
  --mlflow-tracking-uri "$MLFLOW_TRACKING_URI" \
  --mlflow-experiment "$MLFLOW_EXPERIMENT" \
  --mlflow-run-name "$MLFLOW_RUN_NAME" \
  --num-train-epochs 3 \
  --learning-rate 1e-4 \
  --per-device-train-batch-size 1 \
  --per-device-eval-batch-size 1 \
  --gradient-accumulation-steps 8 \
  --max-length 4096 \
  $bf16_flag $fp16_flag

## 6. View MLflow runs inside the notebook

In [ ]:
import mlflow
import pandas as pd

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT)
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

display_columns = [
    col for col in [
        "run_id",
        "status",
        "params.model_name",
        "params.train_examples",
        "params.validation_examples",
        "metrics.train_loss",
        "metrics.train_runtime",
        "tags.mlflow.runName",
    ] if col in runs.columns
]
runs[display_columns]

## 7. Run a sample inference

In [ ]:
import json
from pathlib import Path

import torch
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer

model = AutoPeftModelForCausalLM.from_pretrained(OUTPUT_DIR, torch_dtype="auto", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)

sample_record = json.loads(Path(TRAIN_JSONL).read_text(encoding="utf-8").splitlines()[0])
messages = sample_record["messages"][:2]
prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=512, do_sample=False)

generated = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(generated)

## 8. Optional: download the trained artifacts

In [ ]:
from google.colab import files

!cd /content/training_pipeline && zip -rq hf_syllabus_extractor.zip artifacts/hf_syllabus_extractor mlruns
files.download("/content/training_pipeline/hf_syllabus_extractor.zip")